# 05 HRV Domain And Robustness

Reviewer-facing HRV-domain and robustness readiness/status notebook.


## Setup


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

try:
    os.chdir('/content')
    print('cwd reset:', Path.cwd())
except Exception as exc:
    print(f'cwd reset skipped: {exc}')

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
MIRROR_REVISION_ROOT = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
LOCAL_RUNTIME_ROOT = Path('/content/ecg_ramba_runtime')

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ.setdefault('ECG_RAMBA_LOCAL_ROOT', str(LOCAL_RUNTIME_ROOT))
os.environ.setdefault('ECG_RAMBA_EXTRACT_DIR', str(LOCAL_RUNTIME_ROOT / 'chapman'))
os.environ.setdefault('ECG_RAMBA_USE_CLEAN_CACHE', '0')
os.environ.setdefault('ECG_RAMBA_SAVE_CLEAN_CACHE', '0')

def _run_setup(cmd, cwd=None, check=True):
    print(f'$ {cmd}', flush=True)
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if check and result.returncode:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

def _is_repo(path):
    path = Path(path)
    return (path / '.git').exists() and (path / 'configs' / 'config.py').exists()

def _find_existing_repo():
    candidates = [
        Path.cwd(),
        DRIVE_ROOT / 'ECG-RAMBA',
        DRIVE_ROOT / 'ECG-Ramba',
        DRIVE_ROOT,
        Path('/content/ECG-RAMBA'),
    ]
    for candidate in candidates:
        if _is_repo(candidate):
            return candidate.resolve()
    return None

REPO_DIR = _find_existing_repo()
if REPO_DIR is None:
    clone_target = Path('/content/ECG-RAMBA')
    if clone_target.exists() and not _is_repo(clone_target):
        clone_target = Path(f'/content/ECG-RAMBA-fresh-{int(time.time())}')
    _run_setup(f'git clone --branch {BRANCH} {REPO_URL} "{clone_target}"')
    REPO_DIR = clone_target.resolve()
else:
    print('Found existing repo:', REPO_DIR)

os.chdir(REPO_DIR)

lock_path = REPO_DIR / '.git' / 'index.lock'
if lock_path.exists():
    print('Removing stale git index lock:', lock_path)
    lock_path.unlink()

fetch_result = _run_setup('git fetch origin', cwd=REPO_DIR, check=False)
checkout_result = _run_setup(f'git checkout {BRANCH}', cwd=REPO_DIR, check=False)
if checkout_result.returncode:
    print(f'WARNING: git checkout {BRANCH} failed; continuing on current checkout.')
if fetch_result.returncode == 0:
    pull_result = _run_setup(f'git pull --ff-only --autostash origin {BRANCH}', cwd=REPO_DIR, check=False)
    if pull_result.returncode:
        print('')
        print('=' * 80)
        print('WARNING: git pull failed; continuing with the current checkout.')
        print('If a stale Drive checkout is dirty, use the fresh clone under /content/ECG-RAMBA.')
        print('=' * 80)
else:
    print('WARNING: git fetch failed; continuing with the current checkout.')

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)
_run_setup('git rev-parse --short HEAD', cwd=REPO_DIR, check=False)
_run_setup('git status --short --branch', cwd=REPO_DIR, check=False)

active_revision_root = REPO_DIR / 'reports' / 'revision'
if MIRROR_REVISION_ROOT.exists():
    print('Restoring mirrored revision artifacts from:', MIRROR_REVISION_ROOT)
    shutil.copytree(MIRROR_REVISION_ROOT, active_revision_root, dirs_exist_ok=True)
else:
    print('Mirror not found; notebook will require prior artifacts in the active repo:', MIRROR_REVISION_ROOT)

required_code = [
    'scripts/revision/09_hrv_domain_analysis.py',
    'notebooks/05_hrv_domain_and_robustness.ipynb',
    'docs/revision_plan/experiment_registry.csv',
    'docs/revision_plan/task_board.csv',
]
missing_code = []
for item in required_code:
    path = REPO_DIR / item
    status = 'FOUND' if path.exists() else 'MISSING'
    size = path.stat().st_size if path.exists() else None
    print(f'{item}: {status} size={size}')
    if not path.exists():
        missing_code.append(item)
if missing_code:
    raise FileNotFoundError('Missing required code files after setup: ' + '; '.join(missing_code))

cache_status = {
    'clean_raw_cache': DRIVE_ROOT / 'ecg_data_27c_subject.npz',
    'raw_minirocket_cache': DRIVE_ROOT / 'rocket_raw_N44186_C12_L5000_K10000_S42.npz',
    'hrv36_cache': DRIVE_ROOT / 'hrv36_N44186_C12_L5000.npz',
    'fold_pca_cache_dir': DRIVE_ROOT / 'revision_feature_cache',
    'model_dir_drive': DRIVE_ROOT / 'model',
}
print('cache policy:')
print('  ECG_RAMBA_USE_CLEAN_CACHE =', os.environ.get('ECG_RAMBA_USE_CLEAN_CACHE'))
print('  ECG_RAMBA_SAVE_CLEAN_CACHE=', os.environ.get('ECG_RAMBA_SAVE_CLEAN_CACHE'))
print('  ECG_RAMBA_EXTRACT_DIR     =', os.environ.get('ECG_RAMBA_EXTRACT_DIR'))
print('cache status:')
for name, path in cache_status.items():
    if path.is_dir():
        npz_count = len(list(path.glob('*.npz')))
        pt_count = len(list(path.glob('*.pt')))
        print(f'  {name}: exists=True npz_count={npz_count} pt_count={pt_count} path={path}')
    else:
        print(f'  {name}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')

def run(cmd, check=True, log_path=None, tail_lines=160, cwd=None):
    print(f'$ {cmd}', flush=True)
    run_cwd = Path(cwd) if cwd else Path.cwd()
    if log_path is None:
        log_dir = run_cwd / 'reports' / 'revision' / 'logs'
        log_dir.mkdir(parents=True, exist_ok=True)
        stamp = time.strftime('%Y%m%d_%H%M%S')
        log_path = log_dir / f'notebook_command_{stamp}.log'
    else:
        log_path = Path(log_path)
        if not log_path.is_absolute():
            log_path = run_cwd / log_path
        log_path.parent.mkdir(parents=True, exist_ok=True)

    with log_path.open('w', encoding='utf-8') as log_file:
        proc = subprocess.Popen(
            cmd,
            shell=True,
            cwd=str(run_cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        return_code = proc.wait()

    print(f'Command log: {log_path}')
    if check and return_code:
        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        print(f'Command failed with exit code {return_code}. Last {min(tail_lines, len(lines))} log lines:')
        for line in lines[-tail_lines:]:
            print(line)
        raise subprocess.CalledProcessError(return_code, cmd)
    return subprocess.CompletedProcess(cmd, return_code)


def run_script_if_exists(script_path, command):
    path = Path(script_path)
    if path.exists():
        run(command)
    else:
        print(f'SKIP: {script_path} is not implemented yet.')
        print(f'Planned command: {command}')

print('Setup ready. Continue with Scope And Contracts cell.')


## Scope And Contracts

This notebook now executes the implemented HRV-only and HRV-domain analysis runner. It still blocks robustness stress-test claims because perturbation inference is a separate heavy runner that has not been implemented in this branch. The HRV runner does not load neural checkpoints, perturb signals, or run full-model inference.


In [ ]:
from datetime import datetime, timezone
import math
import numpy as np
import pandas as pd
from IPython.display import display

from scripts.revision.common import git_commit, save_json, sha256_file

RUN_HRV_DOMAIN_ANALYSIS = True
RUN_ROBUSTNESS_STRESS = False

revision_root = Path('reports/revision')
freeze_path = revision_root / 'manifests' / 'oof_freeze_manifest.json'
calibration_path = revision_root / 'metrics' / 'calibration_ci_oof_full_predictions.json'
baseline_summary_path = revision_root / 'metrics' / 'baseline_summary.csv'
component_summary_path = revision_root / 'metrics' / 'component_check_summary.json'
audit_path = revision_root / 'audit_protocol.json'
a0_status_path = revision_root / 'a0_resolution_status.json'
hrv_schema_path = revision_root / 'hrv36_schema.csv'

required_inputs = {
    'oof_freeze_manifest': freeze_path,
    'calibration_ci': calibration_path,
    'baseline_summary': baseline_summary_path,
    'component_check_summary': component_summary_path,
    'audit_protocol': audit_path,
    'a0_resolution_status': a0_status_path,
    'hrv36_schema': hrv_schema_path,
}
for name, path in required_inputs.items():
    print(f'{name:26s}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')
missing = [str(path) for path in required_inputs.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Notebook 01/02/03/04 artifacts are required before notebook 05: ' + '; '.join(missing))

freeze = json.loads(freeze_path.read_text(encoding='utf-8'))
if freeze.get('status') != 'frozen' or freeze.get('manuscript_ready') is not True:
    raise ValueError('OOF freeze manifest must be status=frozen and manuscript_ready=true before HRV/robustness review.')

input_contract = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'inputs': {
        name: {'path': str(path), 'sha256': sha256_file(path), 'size_bytes': path.stat().st_size}
        for name, path in required_inputs.items()
    },
    'freeze_status': freeze.get('status'),
    'freeze_manuscript_ready': freeze.get('manuscript_ready'),
    'allowed_execution': {
        'hrv_only_feature_baseline_training': RUN_HRV_DOMAIN_ANALYSIS,
        'hrv_domain_classifier_training': RUN_HRV_DOMAIN_ANALYSIS,
        'robustness_stress_runner': RUN_ROBUSTNESS_STRESS,
        'full_model_training': False,
        'full_model_inference': False,
        'signal_perturbation': False,
    },
}
save_json(revision_root / 'manifests' / 'hrv_robustness_input_contract.json', input_contract)
print('Input contract validated and written:', revision_root / 'manifests' / 'hrv_robustness_input_contract.json')


## Revision Plan Alignment

This cell binds notebook 05 to HRV-domain and robustness tasks in the tracked revision plan.


In [ ]:
registry = pd.read_csv('docs/revision_plan/experiment_registry.csv')
claims = pd.read_csv('docs/revision_plan/claim_evidence_map.csv')
tasks = pd.read_csv('docs/revision_plan/task_board.csv')

relevant_experiments = registry[registry['experiment_id'].isin(['EXP_HRV', 'EXP_ROBUST'])]
relevant_claims = claims[claims['claim_id'].isin(['C03', 'C04'])]
relevant_tasks = tasks[tasks['id'].isin(['A6', 'B2'])]

display(relevant_experiments)
display(relevant_claims)
display(relevant_tasks)

plan_alignment = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'experiments': relevant_experiments.to_dict(orient='records'),
    'claims': relevant_claims.to_dict(orient='records'),
    'tasks': relevant_tasks.to_dict(orient='records'),
}
save_json(revision_root / 'manifests' / 'hrv_robustness_plan_alignment.json', plan_alignment)
print('Wrote:', revision_root / 'manifests' / 'hrv_robustness_plan_alignment.json')


## HRV Domain Evidence

This cell runs the implemented HRV-only baseline and HRV-domain classifier. The HRV-only baseline uses HRV36 features with the same frozen Chapman OOF folds. The domain classifier uses bounded HRV36 feature caches only; it does not open raw signals or run the neural model.


In [ ]:
hrv_schema = pd.read_csv(hrv_schema_path)
print('HRV schema rows:', len(hrv_schema))
display(hrv_schema.head(10))

hrv_command = (
    'python -u scripts/revision/09_hrv_domain_analysis.py '
    '--threshold 0.5 --n-bins 15 --n-boot 1000 --domain-max-per-domain 2000'
)
if RUN_HRV_DOMAIN_ANALYSIS:
    run(hrv_command, log_path='reports/revision/logs/hrv_domain_analysis.log')
else:
    raise RuntimeError('RUN_HRV_DOMAIN_ANALYSIS is False; HRV evidence will remain blocked.')

hrv_expected_outputs = [
    revision_root / 'predictions' / 'hrv_only_oof_predictions.npz',
    revision_root / 'predictions' / 'hrv_domain_oof_predictions.npz',
    revision_root / 'metrics' / 'hrv_only_baseline_summary.json',
    revision_root / 'metrics' / 'hrv_domain_classifier_summary.json',
    revision_root / 'metrics' / 'hrv_domain_summary.csv',
    revision_root / 'tables' / 'table_hrv_only_class_metrics.csv',
    revision_root / 'tables' / 'table_hrv_domain_classifier_confusion.csv',
    revision_root / 'tables' / 'table_hrv_domain_status.csv',
    revision_root / 'manifests' / 'hrv_domain_analysis_manifest.json',
]
for output in hrv_expected_outputs:
    print(output, 'exists=', output.exists(), 'size=', output.stat().st_size if output.exists() else None)
missing_hrv = [str(output) for output in hrv_expected_outputs if not output.exists()]
if missing_hrv:
    raise FileNotFoundError('Missing HRV analysis outputs: ' + '; '.join(missing_hrv))

hrv_df = pd.read_csv(revision_root / 'metrics' / 'hrv_domain_summary.csv')
hrv_baseline_summary = json.loads((revision_root / 'metrics' / 'hrv_only_baseline_summary.json').read_text(encoding='utf-8'))
hrv_domain_summary = json.loads((revision_root / 'metrics' / 'hrv_domain_classifier_summary.json').read_text(encoding='utf-8'))
print('HRV-only metrics:', json.dumps(hrv_baseline_summary['metrics'], indent=2, sort_keys=True))
print('HRV domain metrics:', json.dumps(hrv_domain_summary['metrics'], indent=2, sort_keys=True))
print('HRV domain interpretation:', hrv_domain_summary.get('interpretation'))
display(hrv_df)


## Robustness Stress-Test Ledger

The revised plan requires perturbation tests. Since no runner exists in this branch, the notebook writes a blocked but reusable robustness table instead of fabricating metrics.


In [ ]:
robustness_rows = []
for analysis_name, perturbation in [
    ('SNR 20 dB noise', 'additive_noise_snr_20db'),
    ('SNR 10 dB noise', 'additive_noise_snr_10db'),
    ('SNR 5 dB noise', 'additive_noise_snr_5db'),
    ('Random 3-lead dropout', 'random_3_lead_dropout'),
    ('V1-V6 dropout', 'precordial_v1_v6_dropout'),
    ('500Hz to 250Hz sampling shift', 'sampling_rate_500_to_250hz'),
]:
    robustness_rows.append({
        'analysis_name': analysis_name,
        'perturbation': perturbation,
        'dataset': 'chapman_oof',
        'status': 'blocked_runner_tbd',
        'clean_metric_reference': str(calibration_path),
        'perturbed_metric_value': math.nan,
        'absolute_delta': math.nan,
        'relative_delta': math.nan,
        'evidence_path': '',
        'blocker': 'No implemented robustness stress-test runner in scripts/revision.',
        'safe_wording': 'Do not claim robustness under this perturbation until the runner is implemented and artifacts are frozen.',
    })

robustness_df = pd.DataFrame(robustness_rows)
robustness_csv = revision_root / 'metrics' / 'robustness_summary.csv'
robustness_table = revision_root / 'tables' / 'table_robustness.csv'
robustness_df.to_csv(robustness_csv, index=False)
robustness_df.to_csv(robustness_table, index=False)
print('Wrote:', robustness_csv)
print('Wrote:', robustness_table)
display(robustness_df)


## Claim Evidence Summary

This JSON is the rebuttal-facing interpretation boundary for HRV and robustness claims.


In [ ]:
hrv_summary_csv = revision_root / 'metrics' / 'hrv_domain_summary.csv'
hrv_df = pd.read_csv(hrv_summary_csv)
hrv_statuses = dict(zip(hrv_df['analysis_name'], hrv_df['status']))
hrv_baseline_summary_path = revision_root / 'metrics' / 'hrv_only_baseline_summary.json'
hrv_domain_classifier_summary_path = revision_root / 'metrics' / 'hrv_domain_classifier_summary.json'
hrv_baseline_summary = json.loads(hrv_baseline_summary_path.read_text(encoding='utf-8'))
hrv_domain_classifier_summary = json.loads(hrv_domain_classifier_summary_path.read_text(encoding='utf-8'))
hrv_complete = (
    hrv_statuses.get('HRV-only baseline') == 'complete'
    and hrv_statuses.get('HRV domain classifier') == 'complete'
)

summary = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'source_freeze_manifest': str(freeze_path),
    'source_calibration': str(calibration_path),
    'hrv_schema': {'path': str(hrv_schema_path), 'sha256': sha256_file(hrv_schema_path)},
    'hrv_status': 'hrv_only_and_domain_classifier_complete_duration_noise_blocked' if hrv_complete else 'blocked_or_incomplete',
    'robustness_status': 'blocked_runner_tbd',
    'hrv_metrics': {
        'hrv_only_roc_auc_macro': hrv_baseline_summary['metrics'].get('roc_auc_macro'),
        'hrv_only_pr_auc_macro': hrv_baseline_summary['metrics'].get('pr_auc_macro'),
        'hrv_only_f1_macro': hrv_baseline_summary['metrics'].get('f1_macro'),
        'domain_roc_auc_ovr_macro': hrv_domain_classifier_summary['metrics'].get('domain_roc_auc_ovr_macro'),
        'domain_balanced_accuracy': hrv_domain_classifier_summary['metrics'].get('balanced_accuracy'),
        'domain_interpretation': hrv_domain_classifier_summary.get('interpretation'),
    },
    'claim_boundaries': [
        {
            'claim_id': 'C03',
            'claim_topic': 'HRV provides complementary rhythm descriptors',
            'current_status': 'partially_supported_by_hrv_only_and_domain_classifier' if hrv_complete else 'not_supported_by_current_artifacts',
            'safe_wording': 'HRV can be reported as a separate feature-only baseline plus domain-sensitivity check; duration/noise HRV sensitivity remains blocked.',
        },
        {
            'claim_id': 'A6/R2C3',
            'claim_topic': 'minimum robustness stress tests',
            'current_status': 'not_supported_by_current_artifacts',
            'safe_wording': 'Robustness tests must be implemented and reported as absolute/relative degradation before any robustness claim.',
        },
    ],
}
summary_path = revision_root / 'metrics' / 'hrv_robustness_component_check_summary.json'
save_json(summary_path, summary)
print('Wrote:', summary_path)
print(json.dumps(summary['hrv_metrics'], indent=2, sort_keys=True))


## Remaining Runner Guard

This cell records the implemented HRV runner and keeps robustness stress testing fail-closed until a dedicated perturbation inference runner exists.


In [ ]:
planned = {
    'HRV-only and domain analysis': 'python scripts/revision/09_hrv_domain_analysis.py',
    'Duration/noise HRV sensitivity': 'python scripts/revision/TBD_hrv_duration_noise_sensitivity.py',
    'Robustness stress tests': 'python scripts/revision/TBD_robustness_stress.py',
}

for name, command in planned.items():
    script = Path(command.split()[1])
    print(f'{name:36s}: implemented={script.exists()} planned_command={command}')

if RUN_ROBUSTNESS_STRESS:
    missing = [name for name, command in planned.items() if not Path(command.split()[1]).exists()]
    if missing:
        raise RuntimeError('Remaining HRV/robustness runners are not implemented yet: ' + ', '.join(missing))
else:
    print('Robustness stress execution disabled. Current notebook keeps robustness claims blocked until a perturbation inference runner exists.')


## Inventory And Stable Mirror

Freeze the revised artifact inventory and publish all reusable outputs to the Drive mirror.


In [ ]:
run(
    'python -u scripts/revision/05_artifact_inventory.py',
    log_path='reports/revision/logs/hrv_robustness_artifact_inventory.log',
)
mirror_root = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
run(
    f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{mirror_root}"',
    log_path='reports/revision/logs/hrv_robustness_mirror_publish.log',
)


## Output Summary


In [ ]:
expected_outputs = [
    revision_root / 'manifests' / 'hrv_robustness_input_contract.json',
    revision_root / 'manifests' / 'hrv_robustness_plan_alignment.json',
    revision_root / 'manifests' / 'hrv_domain_analysis_manifest.json',
    revision_root / 'predictions' / 'hrv_only_oof_predictions.npz',
    revision_root / 'predictions' / 'hrv_domain_oof_predictions.npz',
    revision_root / 'metrics' / 'hrv_only_baseline_summary.json',
    revision_root / 'metrics' / 'hrv_domain_classifier_summary.json',
    revision_root / 'metrics' / 'hrv_domain_summary.csv',
    revision_root / 'tables' / 'table_hrv_domain_status.csv',
    revision_root / 'tables' / 'table_hrv_only_class_metrics.csv',
    revision_root / 'tables' / 'table_hrv_domain_classifier_confusion.csv',
    revision_root / 'metrics' / 'robustness_summary.csv',
    revision_root / 'tables' / 'table_robustness.csv',
    revision_root / 'metrics' / 'hrv_robustness_component_check_summary.json',
    revision_root / 'manifests' / 'artifacts_manifest.json',
    revision_root / 'manifests' / 'artifacts_manifest.csv',
]
for path in expected_outputs:
    print(path, 'exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)

print()
print('Notebook 05 complete. HRV-only/domain evidence is now implemented; robustness claims remain blocked until perturbation inference artifacts exist.')
